# Working with the GPSA BP/RP Spectra

> This Jupyter notebook is in development. This repository was inspired by the use of the original repository GaiaAlerts (Hogg & Sipőcz) and extending its application to the time-domain community. If you use any resources or tools from this project, please cite us and the code used therein.

Author: Andy Tzanidakis

In [3]:
from GaiaAlertsPy import alert as gaap
import numpy as np
from astropy.stats import sigma_clip
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
from matplotlib import rcParams
rcParams['savefig.dpi'] = 550
rcParams['font.size'] = 20
plt.rc('font', family='serif')

/Users/andytzanidakis/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [20]:
from GaiaAlertsPy import alert as gaap
from astropy.io import ascii
from scipy.interpolate import interp1d

In [21]:
# Load BP/RP pixel-wavelength files 
path = "/Users/andytzanidakis/Desktop/desk/astro_research/tda_fun/gaia_spec/"
bp_cal = ascii.read(path + "conv_bp.txt")
rp_cal = ascii.read(path + "conv_rp.txt")

# Perform interpolation of wavelength-pixel solution
w_bp = np.where((bp_cal['col1']>=10) & (bp_cal['col1']<=50))
w_rp = np.where((rp_cal['col1']>=10) & (rp_cal['col1']<=50))
fbp = interp1d(bp_cal['col1'].data[w_bp], bp_cal['col2'].data[w_bp], kind='quadratic')
frp = interp1d(rp_cal['col1'].data[w_rp], rp_cal['col2'].data[w_rp], kind='quadratic')

In [24]:
# Helper functions you'll need 

def pixel_2_nm(px_val, wave='bp', model_bp=fbp, model_rp=frp):
    """Convert from gaia pixels to wavelength solution..."""
    if wave=='bp':
        return model_bp(px_val[::-1])
    elif wave=='rp':
        return model_rp(px_val)

def generate_bprp(_bp, _rp, kind='quadratic', narray=1_000):
    pixel = np.arange(0, 60, step=1)
    xc = np.where((pixel>=10) & (pixel<=50))
    
    x1, y1 = pixel_2_nm(pixel[xc], wave='bp'), _bp[xc]
    x1c = x1<600

    x2, y2 = pixel_2_nm(pixel[xc], wave='rp'), _rp[xc]
    x2c = np.where((x2>700) & (x2<1000))
    
    x, y = np.concatenate([x1[x1c], x2[x2c]]), np.concatenate([y1[x1c], y2[x2c]])
    sf = interp1d(x, y, kind=kind)
    sfx = np.linspace(min(x), max(y), narray) 
    return sf, sf(sfx)

In [23]:
# Load spectrum table with all information including bp/rp spectra
spec_table = gaap.GaiaAlert("Gaia23bvh").query_bprp_history()

# Let's say we want to see what the first spectrum will look like
spec_index = 0 

s1_bp, s1_rp = spec_table['bp'][spec_index], spec_table['rp'][spec_index]




Date,JD,Average Mag.,order,bp,rp,Name
str19,float64,float64,int64,float64[60],float64[60],str9
2014-08-31 21:36:58,2456901.4,17.93,0,-3.1867 .. -0.8133,3.7588 .. 0.2412,Gaia23bvh
